# 🔀 Phase 5: RGB + IR Decision-Level Late Fusion Experiments — Google Colab

This notebook provides the complete GPU-accelerated experimental pipeline to **evaluate, optimize, and benchmark decision-level Late Fusion (Weighted Box Fusion - WBF)** using our independent **Direct Optical RGB** (`yolov5su_rgb_best.pt`) and **Direct Thermal IR** (`yolov5su_ir_best.pt`) models on the official M3FD dataset.

### 🎯 Research Question:
> **Does combining independent RGB and IR decision-level predictions improve detection accuracy over individual modalities, and does IR provide enough complementary information to justify the additional computational cost?**

### 📁 Google Drive Path Mapping:
- **Project Code Path**: `/content/drive/MyDrive/FYP/code` (or `/content/drive/MyDrive/fyp/code`)
- **M3FD Dataset Archive**: `/content/drive/MyDrive/FYP/M3FD_Detection.zip`
- **Checkpoints Folder**: `/content/drive/MyDrive/FYP/code/checkpoints`

### Step 1: Mount Google Drive & Verify GPU Acceleration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU hardware availability
!nvidia-smi

### Step 2: Install Project Dependencies

In [ ]:
%pip install -q kornia thop tabulate PyYAML tqdm opencv-python matplotlib pandas scipy ultralytics

### Step 3: Fast & Robust Environment Setup & Dataset Extraction
Resolves repository paths, extracts the `M3FD_Detection.zip` archive to `/content/m3fd/M3FD_Detection` on fast local NVMe SSD storage, and verifies label formatting.

In [ ]:
# @title ⚙️ Step 3: Fast & Robust Environment Setup & Dataset Extraction
import sys
from pathlib import Path

# Dynamic project path resolution
cand_roots = [
    Path('/content/drive/MyDrive/FYP/code'),
    Path('/content/drive/MyDrive/fyp/code'),
    Path('/content/drive/MyDrive/code'),
    Path('/content/code'),
    Path.cwd()
]
CODE_PATH = next((p for p in cand_roots if (p / 'scripts_AG').exists() or (p / 'TarDAL-main').exists()), Path.cwd())

for p in [CODE_PATH, CODE_PATH / 'scripts_AG', CODE_PATH / 'TarDAL-main']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from stage3_colab_setup import setup_stage3_environment
CODE_PATH, ds_root = setup_stage3_environment()
print(f"\n✅ Environment ready at: {CODE_PATH}")
print(f"✅ Dataset active at: {ds_root}")

### Step 4: Generate & Cache Raw Predictions for RGB & IR YOLOv5su Models
Runs both unimodal models on the official M3FD dataset splits (`val` and `test`) and serializes raw detections $[x_1, y_1, x_2, y_2, \text{conf}, \text{cls}]$ and ground truth annotations into `raw_predictions_{split}.json` on SSD and Google Drive.

> **Why Caching?** Downstream weight optimization sweeps, sensitivity tests, and complementarity analyses run in **$<2\text{ seconds}$** without repeated neural network inference.

In [ ]:
# @title 🚀 Step 4: Extract & Cache Raw Predictions { run: "auto" }
SPLIT_SELECTION = "both" # @param ["both", "val", "test"]
CONF_THRESH = 0.001 # @param {type:"number"}
IOU_THRESH = 0.65 # @param {type:"number"}
BATCH_SIZE = 16 # @param {type:"integer"}
FORCE_REGENERATE = False # @param {type:"boolean"}

gen_script = str(CODE_PATH / 'scripts_AG' / '16_generate_raw_predictions.py')
gen_cmd = (
    f"python -W ignore {gen_script}"
    f" --split {SPLIT_SELECTION}"
    f" --conf {CONF_THRESH}"
    f" --iou {IOU_THRESH}"
    f" --batch_size {BATCH_SIZE}"
)
if FORCE_REGENERATE:
    gen_cmd += " --force"

print(f"\n>>> {gen_cmd}\n")
get_ipython().system(gen_cmd)

### Step 5: Late Fusion Weight Optimization on Validation Split
Performs a systematic grid search on the **Validation set only** ($w_{RGB} \in [0.1, 0.9]$ and LF-1 to LF-5) to identify the optimal $w^*_{RGB}$ and $w^*_{IR}$ that maximizes mAP@50.

- **Score Fusion**: $S_{fusion} = w_{RGB} S_{RGB} + w_{IR} S_{IR}$
- **Box Fusion**: Weighted Box Fusion (WBF)
- **Strict Discipline**: Test set is NOT used for tuning.

In [ ]:
# @title 🧪 Step 5: Optimize Fusion Weights on Validation Split { run: "auto" }
CONF_FILTER = 0.25 # @param {type:"number"}
IOU_MATCH_THRESH = 0.50 # @param {type:"number"}
FUSION_MODE = "wbf" # @param ["wbf", "linear"]

engine_script = str(CODE_PATH / 'scripts_AG' / '17_late_fusion_engine.py')
opt_cmd = (
    f"python -W ignore {engine_script}"
    f" --mode optimize_val"
    f" --conf {CONF_FILTER}"
    f" --iou_match {IOU_MATCH_THRESH}"
    f" --fusion_mode {FUSION_MODE}"
)

print(f"\n>>> {opt_cmd}\n")
get_ipython().system(opt_cmd)

### Step 6: Benchmark Frozen Late Fusion on Official Test Split (840 Pairs)
Locks the optimal validation weights ($w^*_{RGB}, w^*_{IR}$) and benchmarks the frozen system on the unseen **840-image Test set**.

Outputs head-to-head comparison against standalone Direct RGB and Direct IR baselines, reporting overall mAP@50, mAP@50-95, per-class breakdown, and end-to-end hardware latency ($t_{RGB} + t_{IR} + t_{Fusion}$).

In [ ]:
# @title 🔬 Step 6: Benchmark Frozen Model on Test Split { run: "auto" }
MANUAL_W_RGB = 0.60 # @param {type:"number"}
CONF_FILTER = 0.25 # @param {type:"number"}
IOU_MATCH_THRESH = 0.50 # @param {type:"number"}
FUSION_MODE = "wbf" # @param ["wbf", "linear"]

engine_script = str(CODE_PATH / 'scripts_AG' / '17_late_fusion_engine.py')
test_cmd = (
    f"python -W ignore {engine_script}"
    f" --mode eval_test"
    f" --w_rgb {MANUAL_W_RGB}"
    f" --conf {CONF_FILTER}"
    f" --iou_match {IOU_MATCH_THRESH}"
    f" --fusion_mode {FUSION_MODE}"
)

print(f"\n>>> {test_cmd}\n")
get_ipython().system(test_cmd)

### Step 7: Complementarity Analysis & Contingency Decomposition
Categorizes every ground-truth object across 840 test images into 4 mutually exclusive categories:
1. **Both Modalities Succeed** ($\checkmark, \checkmark$): Shared consensus.
2. **RGB Only Succeeds** ($\checkmark, \times$): Structural/color detail.
3. **IR Only Succeeds** ($\times, \checkmark$): **Unique thermal contribution** (empirically proves IR's additive value).
4. **Both Fail** ($\times, \times$): Severe occlusions / extreme distance.

In [ ]:
# @title 🔍 Step 7: Run Complementarity & Venn Decomposition { run: "auto" }
CONF_THRESH = 0.25 # @param {type:"number"}
IOU_HIT_THRESH = 0.50 # @param {type:"number"}

diag_script = str(CODE_PATH / 'scripts_AG' / '18_late_fusion_diagnostics.py')
comp_cmd = (
    f"python -W ignore {diag_script}"
    f" --step complementarity"
    f" --conf {CONF_THRESH}"
    f" --iou {IOU_HIT_THRESH}"
)

print(f"\n>>> {comp_cmd}\n")
get_ipython().system(comp_cmd)

### Step 8: Robustness Sensitivity Sweeps & Sensor Outage Fault-Tolerance
1. **Sensitivity Sweeps**: Tests fusion stability across confidence thresholds $[0.10, 0.25, 0.50]$ and matching IoU $[0.30, 0.50, 0.70]$.
2. **Missing-Modality Experiment**: Simulates sensor dropout ($RGB + \emptyset$ vs $IR + \emptyset$ vs $RGB + IR$) to verify graceful degradation.

In [ ]:
# @title 🛡️ Step 8: Robustness Sweeps & Sensor Dropout { run: "auto" }
W_RGB = 0.60 # @param {type:"number"}

diag_script = str(CODE_PATH / 'scripts_AG' / '18_late_fusion_diagnostics.py')
rob_cmd = f"python -W ignore {diag_script} --step robustness --w_rgb {W_RGB}"
miss_cmd = f"python -W ignore {diag_script} --step missing_modality --w_rgb {W_RGB}"

print(f"\n>>> {rob_cmd}\n")
get_ipython().system(rob_cmd)

print(f"\n>>> {miss_cmd}\n")
get_ipython().system(miss_cmd)

### Step 9: Qualitative Case Studies (5-Panel Multi-Modal Strips)
Generates side-by-side comparative strips targeting the 5 fundamental thesis cases:
- **Case 1**: IR Rescues RGB (Thermal Pedestrian Detection in darkness)
- **Case 2**: RGB Rescues IR (Visible High-Frequency Texture & Lamp Detection)
- **Case 3**: Dual-Modal Consensus (Tightened Localization & High Confidence)
- **Case 4**: Extreme Degradation / Distance (Both Models Missed)
- **Case 5**: Multi-Class Traffic Flow (Car, Bus, Motorcycle Fusion)

In [ ]:
# @title 🎨 Step 9: Generate & Display Qualitative Case Studies { run: "auto" }
W_RGB = 0.60 # @param {type:"number"}
CONF_THRESH = 0.25 # @param {type:"number"}
NUM_DISPLAY = 5 # @param {type:"integer"}

vis_script = str(CODE_PATH / 'scripts_AG' / '19_visualize_late_fusion.py')
vis_cmd = f"python -W ignore {vis_script} --split test --w_rgb {W_RGB} --conf {CONF_THRESH}"
print(f"\n>>> {vis_cmd}\n")
get_ipython().system(vis_cmd)

# Inline Image Display
import glob
from IPython.display import Image, display

vis_dir = CODE_PATH / 'runs' / 'stage5_late_fusion' / 'visualizations'
img_list = sorted(glob.glob(str(vis_dir / '*.jpg')))
print(f"\nDisplaying {min(len(img_list), NUM_DISPLAY)} Qualitative Case Studies:")
for p in img_list[:NUM_DISPLAY]:
    print(f"\nFigure: {Path(p).name}")
    display(Image(filename=p))

### Step 10: Master Quad-Modal Decision Matrix & Final Thesis Conclusion
Assembles the comprehensive architectural comparison table across all four paradigms:
1. **Direct RGB YOLOv5su**
2. **Direct IR YOLOv5su**
3. **Late Fusion (Decision-Level WBF)**
4. **TarDAL Stage 3 Fusion (Feature-Level Task-Driven Attention)**

Answers the final research question: **Does multimodal fusion provide enough accuracy improvement to justify its computational cost?**

In [ ]:
# @title 🧭 Step 10: Run Master Quad-Modal Decision Matrix { run: "auto" }
diag_script = str(CODE_PATH / 'scripts_AG' / '18_late_fusion_diagnostics.py')
dec_cmd = f"python -W ignore {diag_script} --step decision_matrix"
print(f"\n>>> {dec_cmd}\n")
get_ipython().system(dec_cmd)